# MetaPub

## Documentation links
- https://metapub.org
- https://metapub.readthedocs.io/en/latest/examples.html
- https://metapub.readthedocs.io/en/latest/pubmedarticle_properties.html

## Notes
- Using an API key requires env variable to be set
  - `export NCBI_API_KEY="Your_key_here"`
- Currently using local LLM at url in code below (Use Ollama, LMStudio, etc)

In [22]:
from metapub import PubMedFetcher
import requests
import json
from pprint import pprint
import re
import pandas as pd
import os
from umls_python_client import UMLSClient

# API keys
from config import UMLS_API_KEY, NCBI_API_KEY, UMLS_PATH

In [23]:
# modify this endpoint if using different LLM
url = "http://127.0.0.1:1234/v1/chat/completions"

In [24]:
# first patient interaction
# patient_statement_01 = 'I have a headache'
# patient_statement_01 = 'I have a stomach ache that has been there for two weeks and it hurts more after I eat'
# patient_statement_01 = 'I have been sneezing and I have a runny nose for two weeks'
patient_statement_01 = 'I have stomach pain and it hurts more after I eat'
# patient_statement_01 = 'I have a stomach ache and it hurts more after I eat'

In [25]:
# use LLM to pull out potential symptom information only
def query_llm_symptom_check(prompt):
    """
    Queries the LLM API.
    Returns back a list of symptoms based off of statement made
    """
    
    headers = {
        "Content-Type": "application/json"
    }

    # look for symptom information
    prompt = f'''List keywords for medical symptoms in this statement with bullet points 
                and no additional information. Avoid vague singular words like "pain": {prompt}'''

    data = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        # process symptom text into a list
        response = response.json()["choices"][0]["message"]["content"]
        symptoms = re.findall(r'\*\s[^\n]+', response)
        symptoms = [symptom.replace('* ', '') for symptom in symptoms]
        # remove any starting/ending spaces from each item
        symptoms = [symptom.strip() for symptom in symptoms]
        return symptoms
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [26]:
def retrieve_pubmed_articles(symptoms):
    """ Returns a list of 10 pubmed articles relevant to symptoms"""
    # retreive relevant topics to send to PubMed
    symptom_string = symptoms = ', '.join(symptoms)

    # Initialize the fetcher
    fetch = PubMedFetcher()

    # Let user know what's going on
    print(f'Finding articles on:\n{symptom_string}')

    # Search for articles
    pmids = fetch.pmids_for_query(symptom_string, retmax=10)

    article_collection = []

    # Get article details
    for pmid in pmids:
        article = fetch.article_by_pmid(pmid)
        article_collection.append(article)
    
    return article_collection, pmids

In [27]:
def llm_question_formation(symptoms):
    """Queries the LLM API."""
    headers = {
        "Content-Type": "application/json"
    }

    # look for symptom information
    prompt = f'''Take the following list of symptoms to form a question for a 
                patient that is calm, reassuring, and easily understandable with 
                no complex terms. Only include the question text (no explanations): 
                {symptoms}'''

    data = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [28]:
def llm_article_summary(article):
    """
    Queries the LLM API.
    Summarizes article text
    """
    headers = {
        "Content-Type": "application/json"
    }

    # look for symptom information
    prompt = f'Summarize the following text in a single paragraph: {article}'

    data = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        response = response.json()["choices"][0]["message"]["content"]
        return response
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

# Ideas to improve PubMed article retreival

Examples taken from https://metapub.readthedocs.io/en/latest/tutorials.html

- Order articles with more recent dates at top, take the top most recent
- Check full-text availability, create a flag for this (maybe prioritize these too)
- Check access patterns to prioritize use as well

In [29]:
def pubmed_article_logging(pubmed_articles, pmids, article_log, user_id, dialogue_turn):
    # go through all articles
    for pubmed_article, pmid in zip(pubmed_articles, pmids):
        # pull article title, author, abstract, body, llm_summary
        article_source = 'PubMed'
        pubmed_article_id = pmid
        pubmed_article_year = pubmed_article.year
        pubmed_title = pubmed_article.title
        pubmed_authors = ', '.join(pubmed_article.authors)
        pubmed_journal = pubmed_article.journal
        pubmed_abstract = pubmed_article.abstract
        pubmed_citation = pubmed_article.citation_bibtex

        # generate llm summary from abstract (use entire article later)
        pubmed_llm_summary = llm_article_summary(pubmed_abstract)

        # update article_log
        article_log_row = [user_id, dialogue_turn, pubmed_article_id, article_source, 
                            pubmed_article_year, pubmed_title,
                            pubmed_authors, pubmed_journal,
                            pubmed_abstract, 'No article body (TBD)',
                            pubmed_llm_summary, pubmed_citation]
        article_log.loc[len(article_log)] = article_log_row
    return article_log

In [30]:
def umls_retrieval(symptoms):
    # base_uri = 'https://uts-ws.nlm.nih.gov/rest'

    # intialize API
    search_api = UMLSClient(api_key=UMLS_API_KEY).searchAPI

    # basic search
    for symptom in symptoms:
        search_results = search_api.search(
            search_string=symptom,  # The term to search for
            input_type=None,  # None implies search for any input type
            include_obsolete=False,  # Don't include obsolete terms
            include_suppressible=False,  # Don't include suppressible terms
            return_id_type="concept",  # Return UMLS Concept Unique Identifiers (CUIs)
            search_type="words",  # Search using word-based matching
            page_number=1,  # Start from the first page
            page_size=10,  # Limit the result to 10 items per page
            save_to_file=False, # make true if needed later
            file_path=UMLS_PATH
        )
        print(search_results)
    
    # TODO: extract search results
    

In [32]:
# helper function to search through mts_dialogue more effectively
def extract_text(text):
    """Extracts text between 'Symptoms: ' and 'Diagnosis: '."""
    match = re.search(r"Symptoms: (.*?)\nDiagnosis: ", text)
    if match:
        return match.group(1)
    else:
        return None  # Or return an empty string "" if preferred

In [33]:
# helper function to remove numbers from extracted text
def remove_floats(text):
    """Removes float values from the text, keeping only text."""
    if isinstance(text, (int, float)):
        return ''  # Replace numeric values with an empty string
    else:
        return text

In [34]:
def llm_process_mts_dialogue(dialogues, patient_question):
    """
    Looks at sample dialogues on related topic, generates new question
    """
    headers = {
        "Content-Type": "application/json"
    }

    # look for symptom information
    prompt = f'''Use the following dialogue examples and the original patient question
                to generate a question that will give more insight into the patient's
                symptoms. Include the question only in the response. 
                Patient question: {patient_question}. Dialogue examples: {dialogues}'''

    data = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    if response.status_code == 200:
        response = response.json()["choices"][0]["message"]["content"]
        return response
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [35]:
# examine mts dialogue
mts_dialogue = pd.read_csv('data/mts_dialogue/MTS-Dialog-TrainingSet (SDHP).csv')
# do a search in section_text to find matching dialogues with patient symptoms
def doctor_dialogue_mimic(symptoms, patient_question):
    # use regex to extract only symptoms from section_text column
    mts_dialogue['extracted_text'] = mts_dialogue['section_text'].apply(extract_text)
    mts_dialogue['extracted_text'] = mts_dialogue['extracted_text'].apply(remove_floats)

    # take extracted_text closest to patient statement
    dialogues = mts_dialogue[mts_dialogue['extracted_text'].apply(lambda text: any(item in text for item in symptoms))]

    # extract all dialogues and place into a string to feed to LLM
    processed_dialogue = ""
    for index, row in dialogues.iterrows():
        processed_dialogue += f"\ndialogue {index}: {row['dialogue']} "
    # print(processed_dialogue)
    question = llm_process_mts_dialogue(processed_dialogue, patient_question)

    return question

In [ ]:
def surface_patient_information(sample_response=None):
    """ 
    Main function for part 1
    Generates 4 questions for the patient, returns back a dataframe 
    for differential diagnosis
    """
    # generate dataframes
    conversation_log = pd.DataFrame(columns=['user_id', 'dialogue_turn', 
                                             'llm_question', 'patient_answer'])
    symptom_log = pd.DataFrame(columns=['user_id', 'dialogue_turn', 'symptom'])
    article_log = pd.DataFrame(columns=['user_id', 'dialogue_turn', 'article_id',
                                        'article_source', 'article_year',
                                        'article_title', 'article_authors',
                                        'article_journal', 'article_abstract', 
                                        'article_body', 'article_llm_summary',
                                        'article_citation'])
    
    # generate user_id
    # temporary assignment for now
    user_id = 1
    dialogue_turn = 1

    # ask first question
    initial_question = 'How are you feeling today?'
    if sample_response == None:
        initial_response = input(initial_question)
    else:
        initial_response = sample_response
    
    # update conversation log
    conversation_log_row = [user_id, dialogue_turn, 
                            initial_question, initial_response]
    conversation_log.loc[len(conversation_log)] = conversation_log_row

    # find symptoms from initial_response
    symptoms = query_llm_symptom_check(initial_response)

    # update symptom log - 1 row per symptom
    for symptom in symptoms:
        symptom_log.loc[len(symptom_log)] = [user_id, dialogue_turn, symptom]

    # TODO: refine UMLS pull to use for retreival
    # reach out to data sources to gather additional info
    # UMLS
    umls_retrieval(symptoms)

    # use MTS dialogue to find sample conversations related to symtoms
    next_question = doctor_dialogue_mimic(symptoms, initial_response)

    # loop for remaining 3 questions
    for x in range(3):
        dialogue_turn += 1
        next_response = input(next_question)
        # update conversation log
        conversation_log_row = [user_id, dialogue_turn, 
                                next_question, next_response]
        conversation_log.loc[len(conversation_log)] = conversation_log_row

        # find symptoms from initial_response
        symptoms = query_llm_symptom_check(next_response)

        # update symptom log - 1 row per symptom
        for symptom in symptoms:
            symptom_log.loc[len(symptom_log)] = [user_id, dialogue_turn, symptom]

        # use MTS dialogue to find sample conversations related to symtoms
        next_question = doctor_dialogue_mimic(symptoms, next_response)

    # pubmed - save this for the end of questioning to get most relevant articles
    pubmed_articles, pmids = retrieve_pubmed_articles(symptoms)
    # pubmed_article = pubmed_articles[0]
    article_log = pubmed_article_logging(pubmed_articles, pmids, article_log, user_id, dialogue_turn)

    return conversation_log, symptom_log, article_log

In [38]:
# automatic prompt
# conversation_log, symptom_log, article_log = surface_patient_information(patient_statement_01)
# user prompt
conversation_log, symptom_log, article_log = surface_patient_information()

2026-03-04 20:19:53 Brandons-M1-MacBook-Pro.local root[91865] INFO UMLSClient initialized with SearchAPI, SourceAPI, CUIAPI, semanticNetworkAPI and crosswalkAPI
2026-03-04 20:19:53 Brandons-M1-MacBook-Pro.local root[91865] INFO Searching UMLS with parameters: {'string': 'Stomach ache', 'includeObsolete': 'false', 'includeSuppressible': 'false', 'returnIdType': 'concept', 'searchType': 'words', 'partialSearch': 'false', 'pageNumber': 1, 'pageSize': 10, 'apiKey': '08912aa8-0c4b-4434-a3d1-e24ae711d7fb'}
2026-03-04 20:19:53 Brandons-M1-MacBook-Pro.local root[91865] INFO Searching UMLS with parameters: {'string': 'Bloating', 'includeObsolete': 'false', 'includeSuppressible': 'false', 'returnIdType': 'concept', 'searchType': 'words', 'partialSearch': 'false', 'pageNumber': 1, 'pageSize': 10, 'apiKey': '08912aa8-0c4b-4434-a3d1-e24ae711d7fb'}


{
    "pageSize": 10,
    "pageNumber": 1,
    "result": {
        "classType": "searchResults",
        "results": [
            {
                "ui": "C0221512",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0221512",
                "name": "Stomach ache"
            },
            {
                "ui": "C0235309",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0235309",
                "name": "Upset stomach"
            },
            {
                "ui": "C5141843",
                "rootSource": "LNC",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C5141843",
                "name": "I had a bad stomach ache in past 7D"
            },
            {
                "ui": "C5142192",
                "rootSource": "LNC",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C5142192",
            

2026-03-04 20:19:53 Brandons-M1-MacBook-Pro.local root[91865] INFO Searching UMLS with parameters: {'string': 'Constipation', 'includeObsolete': 'false', 'includeSuppressible': 'false', 'returnIdType': 'concept', 'searchType': 'words', 'partialSearch': 'false', 'pageNumber': 1, 'pageSize': 10, 'apiKey': '08912aa8-0c4b-4434-a3d1-e24ae711d7fb'}


{
    "pageSize": 10,
    "pageNumber": 1,
    "result": {
        "classType": "searchResults",
        "results": [
            {
                "ui": "C1291077",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C1291077",
                "name": "Abdominal bloating"
            },
            {
                "ui": "C5980104",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C5980104",
                "name": "Distension of abdomen (finding)"
            },
            {
                "ui": "C4553115",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C4553115",
                "name": "Bloating, CTCAE"
            },
            {
                "ui": "C4302240",
                "rootSource": "SNOMEDCT_US",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C4302240",


2026-03-04 20:19:54 Brandons-M1-MacBook-Pro.local root[91865] INFO Searching UMLS with parameters: {'string': 'Gas', 'includeObsolete': 'false', 'includeSuppressible': 'false', 'returnIdType': 'concept', 'searchType': 'words', 'partialSearch': 'false', 'pageNumber': 1, 'pageSize': 10, 'apiKey': '08912aa8-0c4b-4434-a3d1-e24ae711d7fb'}


{
    "pageSize": 10,
    "pageNumber": 1,
    "result": {
        "classType": "searchResults",
        "results": [
            {
                "ui": "C0009806",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0009806",
                "name": "Constipation"
            },
            {
                "ui": "C0401149",
                "rootSource": "MTH",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0401149",
                "name": "Chronic constipation"
            },
            {
                "ui": "C0729262",
                "rootSource": "SNOMEDCT_US",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0729262",
                "name": "Slow transit constipation"
            },
            {
                "ui": "C0401148",
                "rootSource": "SNOMEDCT_US",
                "uri": "https://uts-ws.nlm.nih.gov/rest/content/2025AB/CUI/C0401148",

In [39]:
conversation_log

,user_id,dialogue_turn,llm_question,patient_answer
0,1,1,How are you feeling today?,"Not great. My stomache hurts. I'm bloated, con..."
1,1,2,"""Could you describe the pain in your stomach m...",It is cramping and mostly in the lower abdomen...
2,1,3,"""Can you describe the cramping - is it sharp, ...",The cramping is sharp. Nothing seems to make t...
3,1,4,What is your activity level like outside of wo...,"Not much activity, just sitting down and walki..."


In [40]:
symptom_log

,user_id,dialogue_turn,symptom
0,1,1,Stomach ache
1,1,1,Bloating
2,1,1,Constipation
3,1,1,Gas
4,1,2,Abdominal cramping
5,1,2,Lower abdominal pain
6,1,3,Cramping
7,1,3,Sharp
8,1,3,Unrelieved
9,1,4,Fatigue


In [41]:
article_log

,user_id,dialogue_turn,article_id,article_source,article_year,article_title,article_authors,article_journal,article_abstract,article_body,article_llm_summary,article_citation


# PubMed article exploration

In [42]:
for index, row in article_log.iterrows():
    pprint(row['article_llm_summary'])